In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
descriptor_path = "resnet_epoch1_topological_descriptors.pt"

descriptor_data = torch.load(
    descriptor_path,
    map_location="cpu"
)

h0_descriptors = descriptor_data["h0_descriptors"].numpy()
h1_descriptors = descriptor_data["h1_descriptors"].numpy()

sample_indices = descriptor_data["sample_indices"]

print("H0 descriptors:", h0_descriptors)
print("H1 descriptors:", h1_descriptors)

print("Number of H0 values:", len(h0_descriptors))
print("Number of H1 values:", len(h1_descriptors))

In [ ]:
representations_path = "resnet_epoch1_vectorized_representations.pt"

representations = torch.load(
    representations_path,
    map_location="cpu"
)

print("Number of blocks:", len(representations))

for i, rep in enumerate(representations):
    print(f"Block {i+1}: {rep.shape}")

In [ ]:
random_seed = 42

generator = torch.Generator().manual_seed(random_seed)

random_order = torch.randperm(
    representations[0].shape[0],
    generator=generator
)

print("First 10 indices:")
print(random_order[:10])

In [ ]:
sample_sizes = np.array([
    50,
    100,
    150,
    200,
    250,
    300
])

print(sample_sizes)

In [ ]:
def lifespan_sum(diagram, alpha=1.0):

    finite_mask = np.isfinite(
        diagram[:, 1]
    )

    finite_pairs = diagram[
        finite_mask
    ]

    if len(finite_pairs) == 0:
        return 0.0

    lifespans = (
        finite_pairs[:, 1]
        - finite_pairs[:, 0]
    )

    return np.sum(
        lifespans ** alpha
    )

In [ ]:
from ripser import ripser
from scipy.stats import linregress

phdim_values = []
beta_values = []
r2_values = []

for block_idx, representation in enumerate(representations):

    print(f"\nProcessing Block {block_idx + 1}/10")

    e_values = []

    for n in sample_sizes:

        indices_n = random_order[:n]

        sample = representation[
            indices_n
        ].numpy()

        result = ripser(
            sample,
            maxdim=0
        )

        h0_diagram = result["dgms"][0]

        e_value = lifespan_sum(
            h0_diagram,
            alpha=1
        )

        e_values.append(e_value)

    e_values = np.array(e_values)

    log_n = np.log10(sample_sizes)
    log_e = np.log10(e_values)

    slope, intercept, r_value, p_value, std_err = linregress(
        log_n,
        log_e
    )

    beta = slope
    r_squared = r_value ** 2

    if beta < 1:
        phdim = 1.0 / (1.0 - beta)
    else:
        phdim = np.nan

    beta_values.append(beta)
    r2_values.append(r_squared)
    phdim_values.append(phdim)

    print(f"Beta:    {beta:.6f}")
    print(f"R²:      {r_squared:.6f}")
    print(f"PHdim:   {phdim:.6f}")

In [ ]:
beta_values = np.array(beta_values)
r2_values = np.array(r2_values)
phdim_values = np.array(phdim_values)

print("PHdim values:")
print(phdim_values)

print("\nBeta values:")
print(beta_values)

print("\nR² values:")
print(r2_values)

In [ ]:
blocks = np.arange(1, 11)

relative_depth = blocks / 10.0

results_df = pd.DataFrame({
    "Block": blocks,
    "Relative_Depth": relative_depth,
    "H0_Descriptor": h0_descriptors,
    "H1_Descriptor": h1_descriptors,
    "Beta": beta_values,
    "R2": r2_values,
    "PHdim": phdim_values
})

print(results_df.to_string(index=False))

In [ ]:
print("Verification")
print("------------")

print(
    "Number of blocks:",
    len(results_df)
)

print(
    "H0 NaN:",
    results_df["H0_Descriptor"].isna().any()
)

print(
    "H1 NaN:",
    results_df["H1_Descriptor"].isna().any()
)

print(
    "PHdim NaN:",
    results_df["PHdim"].isna().any()
)

print(
    "H0 Inf:",
    np.isinf(
        results_df["H0_Descriptor"]
    ).any()
)

print(
    "H1 Inf:",
    np.isinf(
        results_df["H1_Descriptor"]
    ).any()
)

print(
    "PHdim Inf:",
    np.isinf(
        results_df["PHdim"]
    ).any()
)

In [ ]:
results_df.to_csv(
    "resnet_epoch1_layerwise_topology.csv",
    index=False
)

print(
    "Saved: "
    "resnet_epoch1_layerwise_topology.csv"
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    results_df["Relative_Depth"],
    results_df["H0_Descriptor"],
    marker="o"
)

plt.xlabel("Relative depth")
plt.ylabel("H₀ descriptor")
plt.title(
    "H₀ Topological Descriptor Across ResNet Depth"
)

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    results_df["Relative_Depth"],
    results_df["H1_Descriptor"],
    marker="o"
)

plt.xlabel("Relative depth")
plt.ylabel("H₁ descriptor")
plt.title(
    "H₁ Topological Descriptor Across ResNet Depth"
)

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    results_df["Relative_Depth"],
    results_df["PHdim"],
    marker="o"
)

plt.xlabel("Relative depth")
plt.ylabel("PHdim")
plt.title(
    "PHdim Across ResNet Depth"
)

plt.grid(True)
plt.tight_layout()
plt.show()